# Przygotowanie bazy tekstów steganograficznych

Wyciągnięcie stegano tekstów z różnych genereatorów internetowych.
W każdym z tesktów testowych zostanie ukryta losowa wiadomość z listy wiadomości. 
Ten proces zostanie powtórzony dla każdego z generatorów.


In [2]:
# treści wiadomości testowych
import pandas as pd

inputText_df = pd.read_csv("databases/test_review_dataset.csv", usecols=[0])
inputText_df.columns = ['text']

no_of_input_records = 500 #len(inputText_df)


In [4]:
# treści wiadomości do ukrycia
import pandas as pd

catchphrases_df = pd.read_csv("databases/Catchphrase.csv", usecols=[0])
catchphrases_df.columns = ['Catchphrase']

no_of_secret_records = len(catchphrases_df)


Ogólne ustawienia do generowania tekstu

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import time

# Path to the ChromeDriver
driver_path = "C:\Windows\System32\chromedriver-win64\chromedriver.exe"  # Replace with the path to your WebDriver

# driver = webdriver.Chrome(executable_path=driver_path)
driver = webdriver.Chrome()



Strona internetowa: https://www.irongeek.com/i.php?page=security/unicode-steganography-homoglyph-encoder

Sekcja: Unicode Tags Stego

In [6]:
import os

os.makedirs("results", exist_ok=True)

# Open the URL
url = "https://www.irongeek.com/i.php?page=security/unicode-steganography-homoglyph-encoder"  # Replace with the actual URL
driver = webdriver.Chrome()
driver.get(url)

# Allow the page to load 
time.sleep(1)

#Results will be stored here:
results_df = pd.DataFrame(columns=["InputText", "HiddenMessage", "Stegotext"])

try:
    consent_button = driver.find_element(By.CLASS_NAME, "fc-button.fc-cta-consent.fc-primary-button")
    consent_button.click()
except Exception as e:
    print("Can't access the consent button")
for _, input_text in inputText_df.iterrows():
    # Get a random row as hidden message
    random_catchphrase = catchphrases_df.sample(n=1).iloc[0]

    # Find the input tag and type in it
    input_tag = driver.find_element(By.NAME, "coverbox")
    input_tag.clear() 
    input_tag.send_keys(f"{input_text['text']}") 

    # Find the input tag and type in it
    catchphrase_tag = driver.find_element(By.NAME, "inputbox")
    catchphrase_tag.clear()
    catchphrase_tag.send_keys(f"{random_catchphrase['Catchphrase']}") 

    encode_button = driver.find_element(By.NAME, "B1")
    encode_button.click()
    time.sleep(1)
    # Locate by the name attribute
    output_tag = driver.find_element(By.NAME, "stegoputbox")

    # Get the content of the textarea
    stego_content = output_tag.get_attribute("value")

    # Create a new row as a DataFrame
    results_row = pd.DataFrame(
        {
            "InputText": [input_text['text']],        # Wrap scalar in a list
            "HiddenMessage": [random_catchphrase['Catchphrase']],  # Wrap scalar in a list
            "Stegotext": [stego_content],  # Wrap scalar in a list
        }
    )

    # Concatenate the new row to the results_df DataFrame
    results_df = pd.concat([results_df, results_row], ignore_index=True)
    
time.sleep(2)
driver.quit()

# results_df.to_csv("results/irongeek_unicodeTagsStego.csv", encoding="utf-8", index=False)
results_df.to_pickle("results/irongeek_unicodeTagsStego.pkl")


TypeError: NDFrame.to_pickle() got an unexpected keyword argument 'encoding'

Strona internetowa: https://www.irongeek.com/i.php?page=security/unicode-steganography-homoglyph-encoder

Sekcja: More Advanced Unicode Stego:

In [9]:
import os
from selenium.common.exceptions import UnexpectedAlertPresentException, NoAlertPresentException

os.makedirs("results", exist_ok=True)

# Open the URL
url = "https://www.irongeek.com/i.php?page=security/unicode-steganography-homoglyph-encoder"  # Replace with the actual URL
driver = webdriver.Chrome()
driver.get(url)

# Allow the page to load 
time.sleep(1)

#Results will be stored here:
results_df = pd.DataFrame(columns=["InputText", "HiddenMessage", "Stegotext"])

try:
    consent_button = driver.find_element(By.CLASS_NAME, "fc-button.fc-cta-consent.fc-primary-button")
    consent_button.click()
except Exception as e:
    print("Can't access the consent button")
for _, input_text in inputText_df.iterrows():
    # Get a random row as hidden message
    random_catchphrase = catchphrases_df.sample(n=1).iloc[0]

    # Find the input tag and type in it
    textareas = driver.find_elements(By.NAME, "coverbox")  
    input_tag = textareas[1]  
    input_tag.clear() 
    input_tag.send_keys(f"{input_text['text']}") 

    # Find the input tag and type in it
    textareas = driver.find_elements(By.NAME, "inputbox")  
    catchphrase_tag = textareas[1]
    catchphrase_tag.clear()
    catchphrase_tag.send_keys(f"{random_catchphrase['Catchphrase']}") 

    encode_buttons = driver.find_elements(By.NAME, "B1")
    encode_button = encode_buttons[1]
    encode_button.click()
    time.sleep(1)

    try:
        # Check for and handle any unexpected alerts
        alert = driver.switch_to.alert 
        alert.accept()
    except NoAlertPresentException:
        print("No alert present, proceeding...")

    # Locate by the name attribute
    try:
        textareas = driver.find_elements(By.NAME, "stegoputbox")
        if len(textareas) > 1:
            output_tags = textareas[1]  
            stego_content = output_tags.get_attribute("value")  
        else:
            raise Exception("Could not find the expected textarea")

        # Create a new row as a DataFrame
        results_row = pd.DataFrame(
            {
                "InputText": [input_text['text']],        # Wrap scalar in a list
                "HiddenMessage": [random_catchphrase['Catchphrase']],  # Wrap scalar in a list
                "Stegotext": [stego_content],  # Wrap scalar in a list
            }
        )
    except UnexpectedAlertPresentException as e:
        try:
            alert = driver.switch_to.alert
            print("Unexpected Alert Text:", alert.text)
            alert.accept()  # Accept the alert to dismiss it
        except NoAlertPresentException:
            print("Alert disappeared before it could be handled.")
                  
        results_row = pd.DataFrame(
            {
                "InputText": [input_text['text']],        # Wrap scalar in a list
                "HiddenMessage": [random_catchphrase['Catchphrase']],  # Wrap scalar in a list
                "Stegotext": "",  # Wrap scalar in a list
            }
        )
    time.sleep(1)

    

    # Concatenate the new row to the results_df DataFrame
    results_df = pd.concat([results_df, results_row], ignore_index=True)
    
time.sleep(2)
driver.quit()

# results_df.to_csv("results/irongeek_moreAdvancedUnicodeStego.csv", encoding="utf-8", index=False)
results_df.to_pickle("results/irongeek_moreAdvancedUnicodeStego.pkl")

No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert present, proceeding...
No alert

Strona internetowa: https://330k.github.io/misc_tools/unicode_steganography.html

In [ ]:
import os

os.makedirs("results", exist_ok=True)

# Open the URL
url = "https://330k.github.io/misc_tools/unicode_steganography.html"  # Replace with the actual URL
driver = webdriver.Chrome()
driver.get(url)

# Allow the page to load 
time.sleep(1)

#Results will be stored here:
results_df = pd.DataFrame(columns=["InputText", "HiddenMessage", "Stegotext"])

for _, input_text in inputText_df.iterrows():
    # Get a random row as hidden message
    random_catchphrase = catchphrases_df.sample(n=1).iloc[0]

    # Find the input tag and type in it
    input_tag = driver.find_element(By.ID, "input_text")
    input_tag.clear() 
    input_tag.send_keys(f"{input_text['text']}") 

    # Find the input tag and type in it
    catchphrase_tag = driver.find_element(By.ID, "hidden_text")
    catchphrase_tag.clear()
    catchphrase_tag.send_keys(f"{random_catchphrase['Catchphrase']}") 

    encode_button = driver.find_element(By.ID, "encode_text")
    encode_button.click()
    time.sleep(1)
    # Locate by the name attribute
    output_tag = driver.find_element(By.ID, "output_text")

    # Get the content of the textarea
    stego_content = output_tag.get_attribute("value")

    # Create a new row as a DataFrame
    results_row = pd.DataFrame(
        {
            "InputText": [input_text['text']],        # Wrap scalar in a list
            "HiddenMessage": [random_catchphrase['Catchphrase']],  # Wrap scalar in a list
            "Stegotext": [stego_content],  # Wrap scalar in a list
        }
    )

    # Concatenate the new row to the results_df DataFrame
    results_df = pd.concat([results_df, results_row], ignore_index=True)
    
time.sleep(2)
driver.quit()

results_df.to_pickle("results/330k_unicode_steganography.pkl")


Strona internetowa: https://rabian.io/text_watermark/text_watermark_original/index.html

In [7]:
import os

os.makedirs("results", exist_ok=True)

# Open the URL
url = "https://rabian.io/text_watermark/text_watermark_original/index.html"  # Replace with the actual URL
driver = webdriver.Chrome()
driver.get(url)

# Allow the page to load 
time.sleep(1)

#Results will be stored here:
results_df = pd.DataFrame(columns=["InputText", "HiddenMessage", "Stegotext"])

for _, input_text in inputText_df.iterrows():
    # Get a random row as hidden message
    random_catchphrase = catchphrases_df.sample(n=1).iloc[0]

    # Find the input tag and type in it
    input_tag = driver.find_element(By.CLASS_NAME, "visible_text")
    input_tag.clear() 
    input_tag.send_keys(f"{input_text['text']}") 

    # Find the input tag and type in it
    catchphrase_tag = driver.find_element(By.CLASS_NAME, "invisible_text")
    catchphrase_tag.clear()
    catchphrase_tag.send_keys(f"{random_catchphrase['Catchphrase']}") 

    encode_button = driver.find_element(By.CLASS_NAME, "combiner_button")
    encode_button.click()
    time.sleep(2)
    # Locate by the name attribute
    output_tag = driver.find_element(By.CLASS_NAME, "combined_text")

    # Get the content of the textarea
    stego_content = output_tag.get_attribute("value")

    # Create a new row as a DataFrame
    results_row = pd.DataFrame(
        {
            "InputText": [input_text['text']],        # Wrap scalar in a list
            "HiddenMessage": [random_catchphrase['Catchphrase']],  # Wrap scalar in a list
            "Stegotext": [stego_content],  # Wrap scalar in a list
        }
    )

    # Concatenate the new row to the results_df DataFrame
    results_df = pd.concat([results_df, results_row], ignore_index=True)
    
time.sleep(2)
driver.quit()

results_df.to_pickle("results/rabian_text_watermark_original.pkl")

Strona internetowa: https://dblspk.io

In [9]:
import os

os.makedirs("results", exist_ok=True)

# Open the URL
url = "https://dblspk.io"  # Replace with the actual URL
driver = webdriver.Chrome()
driver.get(url)

# Allow the page to load 
time.sleep(1)

#Results will be stored here:
results_df = pd.DataFrame(columns=["InputText", "HiddenMessage", "Stegotext"])

for _, input_text in inputText_df.iterrows():
    # Get a random row as hidden message
    random_catchphrase = catchphrases_df.sample(n=1).iloc[0]

    # Find the input tag and type in it
    input_tag = driver.find_element(By.ID, "out-cover")
    input_tag.clear() 
    input_tag.send_keys(f"{input_text['text']}") 

    # Find the input tag and type in it
    catchphrase_tag = driver.find_element(By.ID, "out-plain")
    catchphrase_tag.clear()
    catchphrase_tag.send_keys(f"{random_catchphrase['Catchphrase']}") 

    time.sleep(2)
    # Locate by the name attribute
    output_tag = driver.find_element(By.ID, "out-cipher")

    # Get the content of the textarea
    stego_content = output_tag.get_attribute("value")

    # Create a new row as a DataFrame
    results_row = pd.DataFrame(
        {
            "InputText": [input_text['text']],        # Wrap scalar in a list
            "HiddenMessage": [random_catchphrase['Catchphrase']],  # Wrap scalar in a list
            "Stegotext": [stego_content],  # Wrap scalar in a list
        }
    )

    # Concatenate the new row to the results_df DataFrame
    results_df = pd.concat([results_df, results_row], ignore_index=True)
    
time.sleep(2)
driver.quit()

results_df.to_pickle("results/doublespeak.pkl")